# [Kaggle] 06 - Evaluate All Models (Paper Format)

Notebook này được tạo ra để **đánh giá tập trung tất cả các model** đã được train và lưu từ các notebook trước (NB01, NB02, NB04, NB05).

Mục tiêu là tạo ra **4 Bảng (Table 3, 4, 5, 6)** theo đúng chuẩn báo cáo khoa học cho từng mô hình để dễ dàng so sánh.

In [1]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict
from IPython.display import display

ASPECTS = ["CAMERA","FEATURES","PERFORMANCE","DESIGN","PRICE",
           "GENERAL","SCREEN","BATTERY","STORAGE","SER&ACC"]
SENTIMENTS = ["POSITIVE","NEUTRAL","NEGATIVE"]

def evaluate_paper_format(pred_spans_list, true_spans_list, aspects_list=ASPECTS, sentiments_list=SENTIMENTS, model_name="Model"):
    """
    Tạo 4 bảng đánh giá theo format paper:
    - Table 3: Overall Aspect / Polarity / Aspect-Polarity (Micro + Macro)
    - Table 4: Per-Aspect P/R/F1
    - Table 5: Per-Sentiment P/R/F1
    - Table 6: F1 matrix (Aspect × Polarity)
    """
    metrics = {
        'Aspect': {'tp': 0, 'fp': 0, 'fn': 0},
        'Polarity': {'tp': 0, 'fp': 0, 'fn': 0},
        'Aspect-Polarity': {'tp': 0, 'fp': 0, 'fn': 0}
    }
    aspect_counts = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})
    sentiment_counts = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})
    aspect_sentiment_counts = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})

    for true_spans, pred_spans in zip(true_spans_list, pred_spans_list):
        # Aspect-Polarity
        t_ap = set([(s[0], s[1], s[2]) for s in true_spans])
        p_ap = set([(s[0], s[1], s[2]) for s in pred_spans])
        
        # Aspect Only
        t_a = set([(s[0].split('#')[0], s[1], s[2]) for s in true_spans])
        p_a = set([(s[0].split('#')[0], s[1], s[2]) for s in pred_spans])
        
        # Polarity Only
        t_p = set([(s[0].split('#')[1] if '#' in s[0] else 'UNK', s[1], s[2]) for s in true_spans])
        p_p = set([(s[0].split('#')[1] if '#' in s[0] else 'UNK', s[1], s[2]) for s in pred_spans])

        def update_counts(true_set, pred_set, overall_dict, class_dicts):
            for span in true_set:
                key = span[0]
                if span in pred_set:
                    overall_dict['tp'] += 1; class_dicts[key]['tp'] += 1
                else:
                    overall_dict['fn'] += 1; class_dicts[key]['fn'] += 1
            for span in pred_set:
                if span not in true_set:
                    key = span[0]
                    overall_dict['fp'] += 1; class_dicts[key]['fp'] += 1

        update_counts(t_ap, p_ap, metrics['Aspect-Polarity'], aspect_sentiment_counts)
        update_counts(t_a, p_a, metrics['Aspect'], aspect_counts)
        update_counts(t_p, p_p, metrics['Polarity'], sentiment_counts)

    def calc_metrics(tp, fp, fn):
        p = tp / (tp + fp) if tp + fp > 0 else 0
        r = tp / (tp + fn) if tp + fn > 0 else 0
        f1 = 2 * p * r / (p + r) if p + r > 0 else 0
        return p * 100, r * 100, f1 * 100

    def calc_macro(counts_dict, keys):
        all_p, all_r, all_f1 = [], [], []
        for k in keys:
            c = counts_dict[k]
            p, r, f1 = calc_metrics(c['tp'], c['fp'], c['fn'])
            all_p.append(p); all_r.append(r); all_f1.append(f1)
        return np.mean(all_p), np.mean(all_r), np.mean(all_f1)

    # === Table 3 ===
    df3 = []
    for task in ['Aspect', 'Polarity', 'Aspect-Polarity']:
        c = metrics[task]
        p_mi, r_mi, f1_mi = calc_metrics(c['tp'], c['fp'], c['fn'])
        if task == 'Aspect':
            keys = aspects_list; counts = aspect_counts
        elif task == 'Polarity':
            keys = sentiments_list; counts = sentiment_counts
        else:
            keys = [f"{a}#{s}" for a in aspects_list for s in sentiments_list]
            counts = aspect_sentiment_counts
        p_ma, r_ma, f1_ma = calc_macro(counts, keys)
        df3.append({'System': task, 'PMicro': p_mi, 'RMicro': r_mi, 'F1Micro': f1_mi,
                    'PMacro': p_ma, 'RMacro': r_ma, 'F1Macro': f1_ma})
    df3 = pd.DataFrame(df3).round(2).set_index('System')

    # === Table 4 ===
    df4 = []
    for a in aspects_list:
        c = aspect_counts[a]
        p, r, f1 = calc_metrics(c['tp'], c['fp'], c['fn'])
        df4.append({'Aspect': a, 'Precision': p, 'Recall': r, 'F1-score': f1})
    df4 = pd.DataFrame(df4).round(2).set_index('Aspect')

    # === Table 5 ===
    df5 = []
    for s in sentiments_list:
        c = sentiment_counts[s]
        p, r, f1 = calc_metrics(c['tp'], c['fp'], c['fn'])
        df5.append({'Sentiment': s, 'Precision': p, 'Recall': r, 'F1-score': f1})
    df5 = pd.DataFrame(df5).round(2).set_index('Sentiment')

    # === Table 6 ===
    df6 = []
    for a in aspects_list:
        row = {'Aspect': a}
        for s in sentiments_list:
            c = aspect_sentiment_counts[f"{a}#{s}"]
            _, _, f1 = calc_metrics(c['tp'], c['fp'], c['fn'])
            row[s.capitalize()] = f1
        df6.append(row)
    df6 = pd.DataFrame(df6).round(2).set_index('Aspect')

    print(f"\n{'='*95}")
    print(f"📊 Table 3: Overall Experimental Results — {model_name}")
    print(f"{'='*95}")
    display(df3)
    
    print(f"\n{'='*95}")
    print(f"📊 Table 4: Per-Aspect P/R/F1")
    print(f"{'='*95}")
    display(df4)
    
    print(f"\n{'='*95}")
    print(f"📊 Table 5: Per-Sentiment P/R/F1")
    print(f"{'='*95}")
    display(df5)
    
    print(f"\n{'='*95}")
    print(f"📊 Table 6: F1-score per Aspect#Polarity")
    print(f"{'='*95}")
    display(df6)

    return df3, df4, df5, df6

## 1. Đánh giá Model Pipeline (ATE từ NB01 + ASC từ NB02)

Ở đây bạn load test set, load model ATE và ASC đã lưu, và gọi hàm `predict_pipeline` để lấy `pred_spans`.

```python
# Ví dụ mẫu (điều chỉnh đường dẫn tới file .pt của bạn):
# ate_model.load_state_dict(torch.load('results/ate/best_ate_model.pt'))
# asc_model.load_state_dict(torch.load('results/asc/best_asc_model.pt'))

# Chạy inference:
# pred_spans_pipeline, true_spans = pipeline_predict_all(test_loader, ate_model, asc_model, ...)

# Đánh giá:
# evaluate_paper_format(pred_spans_pipeline, true_spans, model_name="Pipeline (NB01 + NB02)")
```

In [2]:
# Code load model NB01, NB02 và đánh giá

## 2. Đánh giá Model E2E PhoBERT (từ NB04)

In [3]:
# Code load model E2E PhoBERT (NB04) và đánh giá

# Ví dụ:
# phobert_model.load_state_dict(torch.load('results/e2e/phobert_crf_best.pt'))
# test_res = predict_e2e(phobert_model, test_loader, device)
# pred_spans_phobert = [bio_tags_to_spans(pt, BIO_TAGS, l) for pt, l in zip(test_res['pred_tags'], test_res['lengths'])]
# true_spans_phobert = [bio_tags_to_spans(tt, BIO_TAGS, l) for tt, l in zip(test_res['true_tags'], test_res['lengths'])]

# evaluate_paper_format(pred_spans_phobert, true_spans_phobert, model_name="E2E PhoBERT-CRF (NB04)")

## 3. Đánh giá Model E2E BiGRU Baseline (từ NB05)

In [4]:
# Code load model E2E BiGRU (NB05) và đánh giá

# Ví dụ:
# bigru_model.load_state_dict(torch.load('results/e2e_baseline/bigru_crf_best.pt'))
# test_res = predict_e2e(bigru_model, test_loader, device)
# pred_spans_bigru = [bio_tags_to_spans(pt, BIO_TAGS, l) for pt, l in zip(test_res['pred_tags'], test_res['lengths'])]
# true_spans_bigru = [bio_tags_to_spans(tt, BIO_TAGS, l) for tt, l in zip(test_res['true_tags'], test_res['lengths'])]

# evaluate_paper_format(pred_spans_bigru, true_spans_bigru, model_name="E2E BiGRU-CRF (NB05)")